# From Model to Production: Deploying ML with FastAPI
### Session 2 of 2

---

Welcome again! In this session, we will:
- Load a trained machine learning model  
- Understand how ML models are used in real systems  
- Build a simple REST API using FastAPI  
- Serve real-time fraud predictions
- Consume (send real requests to) your API and see it respond

In **Session 1**,  we explored a real-world fraud detection problem, and then built a machine learning solution.  

We ended by saving two files:
- `fraud_model.pkl` — the trained model
- `scaler.pkl` — the fitted scaler

**Today, we take that model and make it usable by the real world.**

## 1. But first, what Is an API — and Why Do We Need One?

### The Problem
You have a trained model sitting in a `.pkl` file. Right now, only someone who:
- Has Python installed
- Has the right libraries
- Knows how to write the loading code

...can use it. That is not very useful in a real product.

### The Solution: An API
An **API (Application Programming Interface)** is a way for different programs to talk to each other over a network.  
Think of it like a waiter in a restaurant:

```
You (client / consumer)  →  tell the waiter your order  →  Kitchen (server / your model)
Kitchen       →  prepares the food           →  Waiter brings it back to you
```

In our ML case:
```
Mobile app / website  →  sends transaction data  →  Your FastAPI server
FastAPI server        →  runs the model          →  returns fraud prediction
```
The API is how real applications can connect to your model.

### What is REST?
**REST** is a common way of designing APIs using HTTP — the same protocol used by your browser.

The two methods we care about today:

| Method | Purpose | Analogy |
|--------|---------|--------|
| `GET` | Retrieve information | “What’s on the menu?” |
| `POST` | Send data and get a response | Placing your order |

💡We’ll use `POST` to send transaction data and receive predictions.

---

### What is FastAPI?

**FastAPI** is a Python framework for building APIs quickly.

It’s popular because:
- It’s simple to write  
- It automatically creates interactive docs  
- It’s widely used for deploying ML models


💡 **This is how we move from a model in a notebook to something a real system can use.**

## 2. Installation & Setup

In [ ]:
# Install required libraries

# fastapi     — the API framework
# uvicorn     — the web server that runs FastAPI
# pyngrok     — creates a public URL so we can access the API from inside Colab
# nest_asyncio — allows async code (FastAPI in our case) to run inside Jupyter notebooks

!pip install fastapi uvicorn pyngrok nest_asyncio --quiet

In [ ]:
# Core utilities
import joblib            # Load saved model
import numpy as np       # Numerical operations

# FastAPI components
from fastapi import FastAPI
from pydantic import BaseModel

# Running FastAPI inside notebook
import nest_asyncio      # Allows async code to run in notebooks
import uvicorn           # ASGI server to run FastAPI

# Exposing API (Colab workaround)
from pyngrok import ngrok, conf   # Create public URL for API

# Testing & utilities
import requests          # Send HTTP requests to API
import threading         # Run server in background
import json              # Handle JSON data

In [ ]:
# Notebook setup
nest_asyncio.apply()     # Required to run FastAPI inside Jupyter/Colab

## 3. Reload the Model from Session 1

In [ ]:
# First we mount google drive (bring it here so our notebook can see it)

from google.colab import drive
drive.mount('/content/drive')

Make sure the file paths for `fraud_model.pkl` and `scaler.pkl` are correct before running the cell below.

In [ ]:
#  Load the saved model and scaler
fraud_model_path = "/content/drive/MyDrive/pydata-fraud-session/fraud_model.pkl"
scaler_path = "/content/drive/MyDrive/pydata-fraud-session/scaler.pkl"


model  = joblib.load(fraud_model_path)
scaler = joblib.load(scaler_path)

print(f'Model type: {type(model).__name__}')

## 4. Building the API — Step by Step

We will build the API in layers so each piece is clear before we put it all together.

### Step 4.1 — What Does Our API Need to Do?

```
Client sends:    { transaction features }
                         ↓
Our API:         1. Receives the data
                 2. Scales data (just like we did in training)
                 3. Runs the model
                 4. Returns the prediction and fraud probability
                         ↓
Client receives: { "prediction": "Fraud", "fraud_probability": 0.87 }
```

### Step 4.2 — Defining the Input

Our API needs to know what kind of data to expect.

To keep things simple, instead of defining all 30 features individually, we will pass them as a list.


**Note that** in a real production system, we would **define each feature explicitly** (e.g., Time, Amount, V1…V28) for better validation and clarity.

However, for this session:
- We focus on understanding how deployment works
- We simplify the input to reduce complexity

Hoowever, note that the model still expects:
- **Exactly 30 values**
- **In the correct order**

So when sending data to the API, we must ensure:
- The length is correct
- The feature order matches what the model was trained on

In [ ]:
# Input schema
# This class defines exactly what the API expects to receive.

class Transaction(BaseModel):
    features: list  # list of 30 feature values

### Step 4.3 — The Prediction Function

Before building the full API, let's write and test the core prediction logic on its own.

In [ ]:
def predict_fraud(data: Transaction):

    # Convert to array
    features = np.array(data.features).reshape(1, -1)

    # Scale input
    features_scaled = scaler.transform(features)

    # Predict
    prediction  = model.predict(features_scaled)[0]
    probability = model.predict_proba(features_scaled)[0][1]

    return {
        "prediction": "Fraud" if prediction == 1 else "Legitimate",
        "fraud_probability": round(float(probability), 4)
    }

In [ ]:
# Test the prediction function directly (before wrapping in API)
# We create a dummy transaction with all zeros to verify the function works.

dummy = Transaction(features=[0]*30)

result = predict_fraud(dummy)

print(result)
print("\n")

## 5. Building the FastAPI Application

Now we wrap everything into a FastAPI app.

At a high level:
- We create the app  
- Define endpoints (URLs)  
- Each endpoint runs a function  

Think of an endpoint as a “door” into your application.

In [ ]:
# Create FastAPI app
app = FastAPI(
    title="Fraud Detection API",
    description="Deploying our fraud detection model",
    version="1.0.0"
)

#  Root endpoint

# A simple GET endpoint that confirms the API is running.
# Visit the root URL in your browser and you will see this response.
@app.get("/")
def root():
    return {
        "message": "Fraud Detection API is running!",
        "usage": "Send a POST request to /predict with transaction features"
    }

# Prediction endpoint

# Takes input data via POST and returns a prediction
@app.post("/predict")
def predict(data: Transaction):
    return predict_fraud(data)

## 6. Running & Testing the API

We’ve built our API — now let’s bring it to life and use it.

Running the API means starting a server that listens for requests and returns predictions.

### Step 1: Start the API Server

In [ ]:
# Run FastAPI server in the background

def run_server():
    # Start the FastAPI app using Uvicorn on port 8000
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

# Run server in a separate thread so notebook remains usable
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("API server is starting...")

### Step 2: Get Access Link

Because we are running inside Colab, we need a proxy link to access the API.

In [ ]:
from google.colab.output import eval_js  # Allows Python to run JavaScript in Colab (used here to get a public URL)

# Get a public proxy URL for the running server (port 8000)
base_url = eval_js("google.colab.kernel.proxyPort(8000)")

# Direct users to the interactive API interface (/docs)
print("Open this in your browser:")
print(base_url + "/docs")  # Swagger UI for testing endpoints

### Step 3: Open and Use the API

Open the `/docs` link in your browser.

This is FastAPI’s interactive interface.

You can:
- View available endpoints  
- See the expected input format  
- Send requests directly  

---

### Try It Out

1. Find **POST `/predict`**  
2. Click **“Try it out”**  
3. Locate:
```json
{
  "features": [
    "string"
  ]
}
```
4. Replace `["string" ]` with the following sample input:

`[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 100]
}`

5. Click **Execute**

You’ll get a prediction from your model.

In [ ]:
x = [0]*29 + [100]

print(x)

### Testing with Python's Requests

We can also use Python's `requests` library, which lets us make HTTP calls just like a browser or mobile app would.

```
requests.post(url, json=data)   →   sends data to the API
response.json()                 →   reads the API's response

In [ ]:
# Use local server (since we're inside the same environment)
url = "http://127.0.0.1:8000/predict"

# Sample input (must be 30 values)
data = {
    "features": [0]*29 + [100]
}

# Send POST request to API
response = requests.post(url, json=data)

# Print response
print("Status code:", response.status_code)
print("Response from API:")
print(response.json())

print('\n')


### Testing with a Fraud example

In [ ]:
# Let’s take an actual transaction from our dataset that is labeled as fraud.


import pandas as pd

url = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"
df = pd.read_csv(url)

# Split into X and y (features and target)
X = df.drop('Class', axis=1)
y = df['Class']


fraud_sample = X[y == 1].iloc[0]

# Convert to list
fraud_features = fraud_sample.values.tolist()

# Send to API
response = requests.post("http://127.0.0.1:8000/predict", json={"features": fraud_features})

print(response.json())

# 7. What Just Happened?

We:

1. Loaded a trained model  
2. Wrapped it inside an API  
3. Sent data to it  
4. Got predictions back  

 This is how ML models are used in real-world systems

## Real-World Example

Imagine:

- A user makes a transaction  
- The app sends data to this API  
- The API returns:
  - Fraud or Legitimate  
- The system takes action  

All happening in real time

 # 8. A bit of Reality Check

  We Simplified:
- Input format (we used a list instead of full schema)  
- No authentication  
- No database  
- No monitoring  

In real systems, these would have to be added

# 9. Key Takeaways

By the end of this session, you should now understand that:

- A trained model is not useful on its own — it needs to be accessible  
- APIs allow other systems to interact with your model  
- FastAPI makes it easy to turn a model into a usable service  
- We can send real data to our API and get predictions instantly  

---

# 🔄 The Full Journey

Across these two sessions, we’ve gone from:

- Understanding a real-world fraud problem  
- Exploring the data and its challenges (imbalanced data)  
- Building and evaluating a machine learning model  
- To finally turning that model into a live API  

---
 This is the full lifecycle of a machine learning solution:

Problem → Data → Model → Production

---

- Machine learning is not just about building models  
- It’s about making them usable in real-world systems  

---

### Where to Go Next

| Topic | Resource |
|-------|----------|
| FastAPI full tutorial | [fastapi.tiangolo.com/tutorial](https://fastapi.tiangolo.com/tutorial/) |
| Deploying FastAPI to the cloud | [Railway](https://railway.app), [Render](https://render.com), [Google Cloud Run](https://cloud.google.com/run) |
| Model monitoring & drift detection | [Evidently AI](https://www.evidentlyai.com/) |
| Containerising with Docker | [Docker getting started](https://docs.docker.com/get-started/) |
| ML deployment patterns | [Chip Huyen — Designing ML Systems](https://www.oreilly.com/library/view/designing-machine-learning/9781098107956/) |
